In [10]:
import os
import random
import pandas as pd
import numpy as np
import networkx as nx
from gensim.models import Word2Vec
from tqdm import tqdm
import warnings

from node2vec import Node2Vec

from utils import *

# Fix random seeds for reproducibility
np.random.seed(42)
random.seed(42)





In [11]:


# 1) Create directories

os.makedirs("walks", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("embeddings", exist_ok=True)

# 2) Embedding configurations
emb_configs = [
    {"dim": 64,  "walk_length": 40, "num_walks": 10, "p": 1.0, "q": 1.0},
    {"dim": 64,  "walk_length": 20, "num_walks": 10, "p": 1.0, "q": 1.0},
    {"dim": 32,  "walk_length": 40, "num_walks": 20, "p": 0.5, "q": 1.0},
    {"dim": 32,  "walk_length": 40, "num_walks": 10, "p": 1.0, "q": 0.5},
    {"dim": 128, "walk_length": 80, "num_walks": 10, "p": 1.0, "q": 2.0},
]

# 3) Build and validate the heterogeneous graph
G = build_graph(
    dev_room_floor_csv="device_room_floor_cleaned.csv",
    flattened_parquet="flattened_measurements.parquet",
    graph_out_path="hetero_graph.gpickle"
)
validate_graph(G)


[build_graph] Loading existing graph from 'hetero_graph.gpickle' …
[load_graph_pickle] Graph loaded from 'hetero_graph.gpickle'
[validate_graph] 0 isolated nodes (first 5): []
[validate_graph] Node counts by type: {'device': 1596, 'room': 743, 'floor': 8, 'property': 49}


In [12]:
# Load raw CSV and Parquet again (or reuse if already in memory)
df_dev_room = pd.read_csv("device_room_floor_cleaned.csv")
df_flat = pd.read_parquet("flattened_measurements.parquet", columns=["device", "property"])

# 1. Count unique sensor IDs (devices) from both sources
n_sensors_csv  = df_dev_room["device"].nunique()
n_sensors_flat = df_flat["device"].nunique()

# 2. Count unique rooms from CSV
n_rooms = df_dev_room["room"].nunique()

# 3. Compare to graph node counts
device_nodes = sum(1 for _, d in G.nodes(data=True) if d["node_type"] == "device")
room_nodes   = sum(1 for _, d in G.nodes(data=True) if d["node_type"] == "room")

print("\n📊 Sensor & Room Sanity Check:")
print(f"• Unique sensors in CSV         : {n_sensors_csv}")
print(f"• Unique sensors in measurements: {n_sensors_flat}")
print(f"• Device nodes in graph         : {device_nodes}")
print(f"→ Total devices represented     : {len(set(df_dev_room['device']) | set(df_flat['device']))}")
print(f"• Unique rooms in CSV           : {n_rooms}")
print(f"• Room nodes in graph           : {room_nodes}")



📊 Sensor & Room Sanity Check:
• Unique sensors in CSV         : 1593
• Unique sensors in measurements: 443
• Device nodes in graph         : 1596
→ Total devices represented     : 1596
• Unique rooms in CSV           : 743
• Room nodes in graph           : 743


In [13]:

# 4) Loop over each Node2Vec configuration
for cfg in emb_configs:
    dim         = cfg["dim"]
    walk_length = cfg["walk_length"]
    num_walks   = cfg["num_walks"]
    p           = cfg["p"]
    q           = cfg["q"]
    cfg_name    = f"dim{dim}_wl{walk_length}_nw{num_walks}_p{p}_q{q}"

    print(f"\n=== Embedding Config: {cfg_name} ===")

    # 4a) Generate or load walks
    walks_path = f"walks/walks_{cfg_name}.pkl"
    walks = generate_node2vec_walks(
        G,
        dimensions=dim,
        walk_length=walk_length,
        num_walks=num_walks,
        p=p,
        q=q,
        walks_out_path=walks_path,
        workers=4
    )

    # 4b) Train or load the Word2Vec model
    model_path = f"models/node2vec_{cfg_name}.model"
    model = train_node2vec_with_walks(
        walks,
        embedding_dim=dim,
        window_size=5,
        workers=4,
        epochs=5,
        model_out_path=model_path
    )

    # 4c) Extract (or load) device embeddings
    emb_csv = f"embeddings/embeddings_{cfg_name}.csv"
    df_emb = extract_device_embeddings(
        model,
        G,
        embedding_dim=dim,
        out_csv=emb_csv
    )

    print(f"[Notebook 3] Saved embeddings to '{emb_csv}'")


=== Embedding Config: dim64_wl40_nw10_p1.0_q1.0 ===
[generate_node2vec_walks] Loading walks from 'walks/walks_dim64_wl40_nw10_p1.0_q1.0.pkl'
[train_node2vec_with_walks] Loading model from 'models/node2vec_dim64_wl40_nw10_p1.0_q1.0.model'
[extract_device_embeddings] Loading existing embeddings from 'embeddings/embeddings_dim64_wl40_nw10_p1.0_q1.0.csv' …
[Notebook 3] Saved embeddings to 'embeddings/embeddings_dim64_wl40_nw10_p1.0_q1.0.csv'

=== Embedding Config: dim64_wl20_nw10_p1.0_q1.0 ===
[generate_node2vec_walks] Loading walks from 'walks/walks_dim64_wl20_nw10_p1.0_q1.0.pkl'
[train_node2vec_with_walks] Loading model from 'models/node2vec_dim64_wl20_nw10_p1.0_q1.0.model'
[extract_device_embeddings] Loading existing embeddings from 'embeddings/embeddings_dim64_wl20_nw10_p1.0_q1.0.csv' …
[Notebook 3] Saved embeddings to 'embeddings/embeddings_dim64_wl20_nw10_p1.0_q1.0.csv'

=== Embedding Config: dim32_wl40_nw20_p0.5_q1.0 ===
[generate_node2vec_walks] Loading walks from 'walks/walks_dim

In [14]:
import os
import pandas as pd

FEATURE_DIR = "features"     # your Notebook 2 outputs: .pkl per scenario
EMB_DIR     = "embeddings"   # your Notebook 3 outputs: .csv per emb config
COMB_DIR    = "combined"

EMB_CONFS = [
    "dim64_wl40_nw10_p1.0_q1.0",
    "dim64_wl20_nw10_p1.0_q1.0",
    "dim32_wl40_nw20_p0.5_q1.0",
    "dim32_wl40_nw10_p1.0_q0.5",
    "dim128_wl80_nw10_p1.0_q2.0",
]

# now pick up all the .pkl scenarios
SCENARIOS = [
    fn.replace(".pkl","")
    for fn in os.listdir(FEATURE_DIR)
    if fn.endswith(".pkl")
]

os.makedirs(COMB_DIR, exist_ok=True)
for cfg in EMB_CONFS:
    out_subdir = os.path.join(COMB_DIR, cfg)
    os.makedirs(out_subdir, exist_ok=True)

    emb_df = pd.read_csv(f"{EMB_DIR}/embeddings_{cfg}.csv", index_col="device")

    for scen in SCENARIOS:
        stats_df = pd.read_pickle(f"{FEATURE_DIR}/{scen}.pkl").set_index("device")

        combined = stats_df.join(emb_df, how="inner")
        if combined.shape[0] < 3:
            print(f"Skipping {cfg}/{scen}: only {combined.shape[0]} devices")
            continue

        combined.reset_index().to_pickle(f"{out_subdir}/{scen}.pkl")
        print(f"Saved combined features for {cfg}/{scen} → {combined.shape}")

print(" All done!")


Saved combined features for dim64_wl40_nw10_p1.0_q1.0/temp_floor_1__day → (59, 71)
Saved combined features for dim64_wl40_nw10_p1.0_q1.0/temp_humidity_co2_floor_6__day → (50, 71)
Saved combined features for dim64_wl40_nw10_p1.0_q1.0/temp_humidity_co2_floor_7__day → (13, 71)
Saved combined features for dim64_wl40_nw10_p1.0_q1.0/humidity_co2_floor_7__night → (5, 71)
Saved combined features for dim64_wl40_nw10_p1.0_q1.0/temp_humidity_floor_5__fullday → (38, 71)
Saved combined features for dim64_wl40_nw10_p1.0_q1.0/temp_humidity_floor_2__fullday → (38, 71)
Saved combined features for dim64_wl40_nw10_p1.0_q1.0/temp_co2_floor_1__night → (59, 71)
Saved combined features for dim64_wl40_nw10_p1.0_q1.0/temp_humidity_floor_2__night → (37, 71)
Saved combined features for dim64_wl40_nw10_p1.0_q1.0/temp_co2_floor_4__day → (51, 71)
Saved combined features for dim64_wl40_nw10_p1.0_q1.0/temp_co2_floor_5__day → (38, 71)
Saved combined features for dim64_wl40_nw10_p1.0_q1.0/co2_floor_6__night → (44, 71)


In [15]:
'''import os
import random
import pandas as pd
import numpy as np
import networkx as nx
from gensim.models import Word2Vec
from tqdm import tqdm

# ──────────────────────────────────────────────────────────
# (Assume these four functions are already defined above in Notebook 3:)
#   build_graph(dev_room_floor_csv, flattened_parquet, graph_out_path)
#   generate_all_walks(G, num_walks, walk_length, walks_out_path)
#   train_node2vec(all_walks, embedding_dim, window_size, workers, epochs, model_out_path)
#   extract_device_embeddings(model, G, embedding_dim, out_csv)
# ──────────────────────────────────────────────────────────

# 1) Create necessary directories
os.makedirs("walks", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("embeddings", exist_ok=True)

# 2) Define embedding configurations to try
emb_configs = [
    {"dim": 64,  "walk_length": 40, "num_walks": 10, "p": 1.0, "q": 1.0},
    {"dim": 64,  "walk_length": 20, "num_walks": 10, "p": 1.0, "q": 1.0},
    {"dim": 32,  "walk_length": 40, "num_walks": 20, "p": 0.5, "q": 1.0},
    {"dim": 32,  "walk_length": 40, "num_walks": 10, "p": 1.0, "q": 0.5},
    {"dim": 128, "walk_length": 80, "num_walks": 10, "p": 1.0, "q": 2.0},
]

# 3) Build (or load) the heterogeneous graph once
G = build_graph(
    dev_room_floor_csv="device_room_floor_cleaned.csv",
    flattened_parquet="flattened_measurements.parquet",
    graph_out_path="hetero_graph.gpickle"
)

# 4) Loop over each embedding configuration
for cfg in emb_configs:
    dim         = cfg["dim"]
    walk_length = cfg["walk_length"]
    num_walks   = cfg["num_walks"]
    p           = cfg["p"]
    q           = cfg["q"]
    cfg_name    = f"dim{dim}_wl{walk_length}_nw{num_walks}_p{p}_q{q}"

    print(f"\n=== Embedding Config: {cfg_name} ===")

    # 4a) Generate or load random walks
    walks_path = f"walks/walks_{cfg_name}.pkl"
    all_walks = generate_all_walks(
        G,
        num_walks=num_walks,
        walk_length=walk_length,
        walks_out_path=walks_path
    )

    # 4b) Train or load the Word2Vec model
    model_path = f"models/node2vec_{cfg_name}.model"
    model = train_node2vec(
        all_walks,
        embedding_dim=dim,
        window_size=5,
        workers=4,
        epochs=5,
        model_out_path=model_path
    )

    # 4c) Extract (or load) device embeddings into a CSV
    out_csv = f"embeddings/embeddings_{cfg_name}.csv"
    df_emb = extract_device_embeddings(
        model,
        G,
        embedding_dim=dim,
        out_csv=out_csv
    )

    print(f"[Notebook 3] Saved embeddings to '{out_csv}'")'''

'import os\nimport random\nimport pandas as pd\nimport numpy as np\nimport networkx as nx\nfrom gensim.models import Word2Vec\nfrom tqdm import tqdm\n\n# ──────────────────────────────────────────────────────────\n# (Assume these four functions are already defined above in Notebook 3:)\n#   build_graph(dev_room_floor_csv, flattened_parquet, graph_out_path)\n#   generate_all_walks(G, num_walks, walk_length, walks_out_path)\n#   train_node2vec(all_walks, embedding_dim, window_size, workers, epochs, model_out_path)\n#   extract_device_embeddings(model, G, embedding_dim, out_csv)\n# ──────────────────────────────────────────────────────────\n\n# 1) Create necessary directories\nos.makedirs("walks", exist_ok=True)\nos.makedirs("models", exist_ok=True)\nos.makedirs("embeddings", exist_ok=True)\n\n# 2) Define embedding configurations to try\nemb_configs = [\n    {"dim": 64,  "walk_length": 40, "num_walks": 10, "p": 1.0, "q": 1.0},\n    {"dim": 64,  "walk_length": 20, "num_walks": 10, "p": 1.0

In [16]:
'''import os


# ─────────────────────────────────────────────────────────────
# 0) Hyperparameters & Paths (tweak as needed)
# ─────────────────────────────────────────────────────────────
DEV_ROOM_CSV       = "device_room_floor_cleaned.csv"
FLAT_PARQUET       = "flattened_measurements.parquet"
GRAPH_PATH         = "hetero_graph.gpickle"
WALKS_PATH         = "all_walks.pkl"
MODEL_PATH         = "node2vec.model"
EMB_OUT_CSV        = "device_embeddings_node2vec.csv"

NUM_WALKS_PER_NODE = 10     # e.g. 10
WALK_LENGTH         = 40    # e.g. 40
EMBED_DIM           = 64    # e.g. 64
WINDOW_SIZE        = 5      # e.g. 5
WORKERS            = 4      # CPU cores for Word2Vec
EPOCHS             = 5      # e.g. 5
# ─────────────────────────────────────────────────────────────

# 1) Build or load the heterogeneous graph
G = build_graph(
    dev_room_floor_csv=DEV_ROOM_CSV,
    flattened_parquet=FLAT_PARQUET,
    graph_out_path=GRAPH_PATH
)

# 2) Generate (or load) random walks
all_walks = generate_all_walks(
    G,
    num_walks=NUM_WALKS_PER_NODE,
    walk_length=WALK_LENGTH,
    walks_out_path=WALKS_PATH
)

# 3) Train (or load) the Word2Vec model
model = train_node2vec(
    all_walks,
    embedding_dim=EMBED_DIM,
    window_size=WINDOW_SIZE,
    workers=WORKERS,
    epochs=EPOCHS,
    model_out_path=MODEL_PATH
)

# 4) Extract (or load) device embeddings and save to CSV
df_device_emb = extract_device_embeddings(
    model,
    G,
    embedding_dim=EMBED_DIM,
    out_csv=EMB_OUT_CSV
)

print("\n[run_notebook3] Completed. Device embeddings shape:", df_device_emb.shape)
print(f"[run_notebook3] Embeddings saved at '{EMB_OUT_CSV}'")'''

'import os\n\n\n# ─────────────────────────────────────────────────────────────\n# 0) Hyperparameters & Paths (tweak as needed)\n# ─────────────────────────────────────────────────────────────\nDEV_ROOM_CSV       = "device_room_floor_cleaned.csv"\nFLAT_PARQUET       = "flattened_measurements.parquet"\nGRAPH_PATH         = "hetero_graph.gpickle"\nWALKS_PATH         = "all_walks.pkl"\nMODEL_PATH         = "node2vec.model"\nEMB_OUT_CSV        = "device_embeddings_node2vec.csv"\n\nNUM_WALKS_PER_NODE = 10     # e.g. 10\nWALK_LENGTH         = 40    # e.g. 40\nEMBED_DIM           = 64    # e.g. 64\nWINDOW_SIZE        = 5      # e.g. 5\nWORKERS            = 4      # CPU cores for Word2Vec\nEPOCHS             = 5      # e.g. 5\n# ─────────────────────────────────────────────────────────────\n\n# 1) Build or load the heterogeneous graph\nG = build_graph(\n    dev_room_floor_csv=DEV_ROOM_CSV,\n    flattened_parquet=FLAT_PARQUET,\n    graph_out_path=GRAPH_PATH\n)\n\n# 2) Generate (or load) random 